# APOBaseJ

> Base class containing the core methods of CRLD agents learning under partial observability (JAX variant).

In [ ]:
#| default_exp Agents/POBaseJ

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import *

In [ ]:
#| hide
%load_ext autoreload
%autoreload 2

In [ ]:
#| export
import jax
import numpy as np
import itertools as it

import jax.numpy as jnp
from jax import jit, vmap
from functools import partial

from fastcore.utils import *

from pyCRLD.Agents.BaseJ import abaseJ
from pyCRLD.Utils.HelpersJ import *

In [ ]:
#| export
class aPObaseJ(abaseJ):
    """
    Base class for
    deterministic policy-average (/ memory mean field) independent (multi-agent)
    temporal-difference reinforcement learning with partial observability.

    To be used as a base for both, value and policy dynamics.
    """

    def __init__(self,
                 TransitionTensor,
                 RewardTensor,
                 ObservationTensor,
                 DiscountFactors,
                 use_prefactor=False,
                 opteinsum=True,
                 **kwargs):
        """
        Parameters
        ----------
        TransitionTensor : transition model of the environment
        RewardTensor : reward model of the environment
        DiscountFactors : the agents' discount factors
        use_prefactor : use the 1-DiscountFactor prefactor (default: False)
        opteinsum : keyword argument to optimize einsum methods (default: True)
        """
        R = jnp.array(RewardTensor)
        T = jnp.array(TransitionTensor)
        O = jnp.array(ObservationTensor)

        # number of agents
        N = R.shape[0]
        assert len(T.shape[1:-1]) == N, "Inconsistent number of agents"
        assert len(R.shape[2:-1]) == N, "Inconsistent number of agents"
        assert O.shape[0] == N, "Inconsistent number of agents"

        # number of actions for each agent
        M = T.shape[1]
        assert np.allclose(T.shape[1:-1], M), 'Inconsisten number of actions'
        assert np.allclose(R.shape[2:-1], M), 'Inconsisten number of actions'

        # number of states
        Z = T.shape[0]
        assert T.shape[-1] == Z, 'Inconsisten number of states'
        assert R.shape[-1] == Z, 'Inconsisten number of states'
        assert R.shape[1] == Z, 'Inconsisten number of states'
        assert O.shape[1] == Z, 'Inconsistent number of states'

        # number of observations
        Q = O.shape[-1]

        self.R, self.T, self.O = R, T, O
        self.N, self.M, self.Z, self.Q = N, M, Z, Q

        # discount factors
        self.gamma = make_variable_vectorJ(DiscountFactors, N)

        # use (1-DiscountFactor) prefactor to have values on scale of rewards
        self.pre = 1 - self.gamma if use_prefactor else jnp.ones(N)
        self.use_prefactor = use_prefactor

        # 'load' the other agents actions summation tensor for speed
        self.Omega = self._OtherAgentsActionsSummationTensor()

        # use optimized einsum method
        self.opti = opteinsum

In [ ]:
#| export
@partial(jit, static_argnums=0)
def Xisa(self: aPObaseJ, X):
    """
    Compute state-action policy given the current observation-action policy
    """
    i = 0; a = 1; s = 2; o = 4  # variables
    args = [self.O, [i, s, o], X, [i, o, a], [i, s, a]]
    Xisa = jnp.einsum(*args, optimize=self.opti)

    # assert np.allclose(Xisa.sum(-1), 1.0), 'Not a policy. Must not happen!'
    return Xisa
aPObaseJ.Xisa = partial(jit, static_argnums=0)(Xisa)

In [ ]:
#| export
@partial(jit, static_argnums=0)
def Tss_po(self: aPObaseJ, X):
    """Compute average transition model Tss given policy X"""
    Xisa_ = self.Xisa(X)
    return abaseJ.Tss(self, Xisa_)
aPObaseJ.Tss = partial(jit, static_argnums=0)(Tss_po)

In [ ]:
#| export
@patch
def Bios(self: aPObaseJ, X):
    """
    Compute 'belief' that environment is in state s given agent i
    observes observation o.
    """
    pS = self.Ps(X)
    return self._bios(X, pS)

@partial(jit, static_argnums=0)
def _bios(self: aPObaseJ, X, pS):
    i, s, o = 0, 1, 2  # variables

    b = jnp.einsum(self.O, [i, s, o], pS, [s], [i, s, o], optimize=self.opti)
    bsum = b.sum(axis=1, keepdims=True)
    bsum = bsum + (bsum == 0)  # to avoid dividing by zero
    Biso = b / bsum
    Bios = jnp.swapaxes(Biso, 1, -1)

    return Bios
aPObaseJ._bios = partial(jit, static_argnums=0)(_bios)

@partial(jit, static_argnums=0)
def fast_BiosJ(self: aPObaseJ, X):
    """
    Compute 'belief' that environment is in state s given agent i
    observes observation o.
    """
    i, s, o = 0, 1, 2  # variables
    # pS = self.statedist(X) # from full obs base (requires Tss from above)
    pS = self._jaxPs(X, self._last_statedist) if hasattr(self, '_last_statedist') \
        else self._jaxPs(X, jnp.ones(self.Z) / self.Z)

    b = jnp.einsum(self.O, [i, s, o], pS, [s], [i, s, o], optimize=self.opti)
    bsum = b.sum(axis=1, keepdims=True)
    bsum = bsum + (bsum == 0)  # to avoid dividing by zero
    Biso = b / bsum
    Bios = jnp.swapaxes(Biso, 1, -1)

    return Bios
aPObaseJ.fast_Bios = partial(jit, static_argnums=0)(fast_BiosJ)

In [ ]:
#| export
@partial(jit, static_argnums=0)
def Tioo(self: aPObaseJ, X, Bios=None, Xisa=None):
    """Compute average transition model Tioo, given joint policy X"""
    # For speed up
    Bios = self.fast_Bios(X) if Bios is None else Bios
    Xisa = self.Xisa(X) if Xisa is None else Xisa

    i = 0; s = 1; s_ = 2; o = 3; o_ = 4; b2d = list(range(5, 5 + self.N))

    Y4einsum = list(it.chain(*zip(Xisa,
                                  [[s, b2d[a]] for a in range(self.N)])))

    args = [Bios, [i, o, s]] + Y4einsum + [self.T, [s] + b2d + [s_],
            self.O, [i, s_, o_], [i, o, o_]]
    return jnp.einsum(*args, optimize=self.opti)
aPObaseJ.Tioo = partial(jit, static_argnums=0)(Tioo)

In [ ]:
#| export
@partial(jit, static_argnums=0)
def Tioao(self: aPObaseJ, X, Bios=None, Xisa=None):
    """Compute average transition model Tioao, given joint policy X"""
    # For speed up
    Bios = self.fast_Bios(X) if Bios is None else Bios
    Xisa = self.Xisa(X) if Xisa is None else Xisa

    i = 0; a = 1; s = 2; s_ = 3; o = 4; o_ = 5;
    j2k = list(range(6, 6 + self.N - 1))
    b2d = list(range(6 + self.N - 1, 6 + self.N - 1 + self.N))
    e2f = list(range(5 + 2 * self.N, 5 + 2 * self.N + self.N - 1))

    sumsis = [[j2k[l], s, e2f[l]] for l in range(self.N - 1)]
    otherY = list(it.chain(*zip((self.N - 1) * [Xisa], sumsis)))

    args = [self.Omega, [i] + j2k + [a] + b2d + e2f,
            Bios, [i, o, s]] + otherY + [self.T, [s] + b2d + [s_],
            self.O, [i, s_, o_], [i, o, a, o_]]
    return jnp.einsum(*args, optimize=self.opti)
aPObaseJ.Tioao = partial(jit, static_argnums=0)(Tioao)

In [ ]:
#| export
@partial(jit, static_argnums=0)
def Rioa(self: aPObaseJ, X, Bios=None, Xisa=None):
    """Compute average reward Riosa, given joint policy X """
    # For speed up
    Bios = self.fast_Bios(X) if Bios is None else Bios
    Xisa = self.Xisa(X) if Xisa is None else Xisa

    i = 0; a = 1; s = 2; s_ = 3; o = 4
    j2k = list(range(5, 5 + self.N - 1))
    b2d = list(range(5 + self.N - 1, 5 + self.N - 1 + self.N))
    e2f = list(range(4 + 2 * self.N, 4 + 2 * self.N + self.N - 1))

    sumsis = [[j2k[l], s, e2f[l]] for l in range(self.N - 1)]
    otherY = list(it.chain(*zip((self.N - 1) * [Xisa], sumsis)))

    args = [self.Omega, [i] + j2k + [a] + b2d + e2f, Bios, [i, o, s]] + \
            otherY + [self.T, [s] + b2d + [s_], self.R, [i, s] + b2d + [s_],
            [i, o, a]]

    return jnp.einsum(*args, optimize=self.opti)
aPObaseJ.Rioa = partial(jit, static_argnums=0)(Rioa)

In [ ]:
#| export
@partial(jit, static_argnums=0)
def Rio(self: aPObaseJ, X, Bios=None, Xisa=None, Rioa=None):
    """Compute average reward Rio, given joint policy X"""
    # For speed up
    if Rioa is None:  # Compute Rio from scratch
        Bios = self.fast_Bios(X) if Bios is None else Bios
        Xisa = self.Xisa(X) if Xisa is None else Xisa

        # Variables
        # agent i, state s, next state s_, observation o,  # all actions
        i = 0; s = 1; s_ = 2; o = 3; b2d = list(range(4, 4 + self.N))

        Y4einsum = list(it.chain(*zip(Xisa,
                                [[s, b2d[a]] for a in range(self.N)])))

        args = [Bios, [i, o, s]] + Y4einsum + [self.T, [s] + b2d + [s_],
                self.R, [i, s] + b2d + [s_], [i, o]]
        return jnp.einsum(*args, optimize=self.opti)

    else:  # Compute Rio based on Rioa (should be faster by factor 20)
        i = 0; o = 1; a = 2  # Variables
        args = [X, [i, o, a], Rioa, [i, o, a], [i, o]]
        return jnp.einsum(*args, optimize=self.opti)
aPObaseJ.Rio = partial(jit, static_argnums=0)(Rio)

In [ ]:
#| export
@partial(jit, static_argnums=0)
def Vio(self: aPObaseJ, X,
        Rio=None, Tioo=None, Bios=None, Xisa=None, Rioa=None,
        gamma=None):
    """Compute average observation values Vio, given joint policy X"""
    gamma = self.gamma if gamma is None else gamma

    # For speed up
    Bios = self.fast_Bios(X) if Bios is None else Bios
    Xisa = self.Xisa(X) if Xisa is None else Xisa
    Rio = self.Rio(X, Bios=Bios, Xisa=Xisa, Rioa=Rioa) if Rio is None \
        else Rio
    Tioo = self.Tioo(X, Bios=Bios, Xisa=Xisa) if Tioo is None \
        else Tioo

    i = 0; o = 1; op = 2  # Variables
    n = jnp.newaxis
    Mioo = jnp.eye(self.Q)[n, :, :] - gamma[:, n, n] * Tioo
    invMioo = jnp.linalg.inv(Mioo)

    return self.pre[:, n] * jnp.einsum(invMioo, [i, o, op], Rio, [i, op],
                                       [i, o], optimize=self.opti)
aPObaseJ.Vio = partial(jit, static_argnums=0)(Vio)

In [ ]:
#| export
@partial(jit, static_argnums=0)
def Qioa(self: aPObaseJ, X, Rioa=None, Vio=None, Tioao=None, Bios=None, Xisa=None,
         gamma=None):
    """Compute average observation-action values Qioa, given joint policy X"""
    gamma = self.gamma if gamma is None else gamma
    # For speed up
    Rioa = self.Rioa(X, Bios=Bios, Xisa=Xisa) if Rioa is None \
        else Rioa
    Vio = self.Vio(X, Bios=Bios, Xisa=Xisa, Rioa=Rioa) if Vio is None \
        else Vio
    Tioao = self.Tioao(X, Bios=Bios, Xisa=Xisa) if Tioao is None \
        else Tioao

    nextQioa = jnp.einsum(Tioao, [0, 1, 2, 3], Vio, [0, 3], [0, 1, 2],
                         optimize=self.opti)
    n = jnp.newaxis
    return self.pre[:, n, n] * Rioa + gamma[:, n, n] * nextQioa
aPObaseJ.Qioa = partial(jit, static_argnums=0)(Qioa)

In [ ]:
#| export
@partial(jit, static_argnums=0)
def Ri_po(self: aPObaseJ, X):
    """Compute average reward Ri, given joint policy X"""
    i, o = 0, 1
    return jnp.einsum(self.obsdist(X), [i, o], self.Rio(X), [i, o], [i])
aPObaseJ.Ri = partial(jit, static_argnums=0)(Ri_po)

In [ ]:
#| export
@patch
def obsdist(self: aPObaseJ, X):
    """Compute stationary observation distribution, given joint policy X."""
    pO0 = jnp.ones((self.N, self.Q)) / self.Q
    return self._jobsdist(X, pO0)

In [ ]:
#| export
@partial(jit, static_argnums=0)
def _jobsdist(self: aPObaseJ, X, pO0, rndkey=42):
    """Compute stationary observation distribution for each agent."""
    Tioo_ = self.Tioo(X)  # shape: (N, Q, Q)

    def single_agent_dist(Tioo_i, pO0_i):
        pO = compute_stationarydistributionJ(Tioo_i)  # shape: (Q, Q)
        nrS = jnp.where(pO.mean(0) != -10, 1, 0).sum()

        def single_dist(_pO):
            return jnp.max(jnp.where(_pO.mean(0) != -10,
                                      jnp.arange(_pO.shape[0]), -1))

        def multi_dist(_pO):
            return jnp.argmin(jnp.linalg.norm(_pO.T - pO0_i, axis=-1))

        ix = jax.lax.cond(nrS == 1, single_dist, multi_dist, pO)
        return pO[:, ix]

    return jax.vmap(single_agent_dist)(Tioo_, pO0)
aPObaseJ._jobsdist = partial(jit, static_argnums=0)(_jobsdist)

In [ ]:
#| export
# Additional state-based averages (delegate to abaseJ via Xisa)
@partial(jit, static_argnums=0)
def Tisas_po(self: aPObaseJ, X):
    """Compute average transition model Tisas, given joint policy X"""
    Xisa_ = self.Xisa(X)
    return abaseJ.Tisas(self, Xisa_)
aPObaseJ.Tisas = partial(jit, static_argnums=0)(Tisas_po)

@partial(jit, static_argnums=0)
def Risa_po(self: aPObaseJ, X):
    """Compute average reward Risa, given joint policy X"""
    Xisa_ = self.Xisa(X)
    return abaseJ.Risa(self, Xisa_)
aPObaseJ.Risa = partial(jit, static_argnums=0)(Risa_po)

@partial(jit, static_argnums=0)
def Ris_po(self: aPObaseJ, X, Risa=None):
    """Compute average reward Ris, given joint policy X"""
    Xisa_ = self.Xisa(X)
    return abaseJ.Ris(self, Xisa_, Risa=Risa)
aPObaseJ.Ris = partial(jit, static_argnums=0)(Ris_po)

@partial(jit, static_argnums=0)
def Vis_po(self: aPObaseJ, X, Ris=None, Tss=None, Risa=None):
    """Compute average state values Vis, given joint policy X"""
    Xisa_ = self.Xisa(X)
    Ris_ = self.Ris(X) if Ris is None else Ris
    Tss_ = self.Tss(X) if Tss is None else Tss
    return abaseJ.Vis(self, Xisa_, Ris=Ris_, Tss=Tss_, Risa=Risa)
aPObaseJ.Vis = partial(jit, static_argnums=0)(Vis_po)

@partial(jit, static_argnums=0)
def Qisa_po(self: aPObaseJ, X, Risa=None, Vis=None, Tisas=None):
    """Compute average state-action values Qisa, given joint policy X"""
    Xisa_ = self.Xisa(X)
    Risa_ = self.Risa(X) if Risa is None else Risa
    Vis_ = self.Vis(X) if Vis is None else Vis
    Tisas_ = self.Tisas(X) if Tisas is None else Tisas
    return abaseJ.Qisa(self, Xisa_, Risa=Risa_, Vis=Vis_, Tisas=Tisas_)
aPObaseJ.Qisa = partial(jit, static_argnums=0)(Qisa_po)

## Tests

In [ ]:
# Minimal test of aPObaseJ using identity observation tensor (= full observability)
from pyCRLD.Environments.SocialDilemmaJ import SocialDilemmaJ

env = SocialDilemmaJ(R=3., T=5., S=0., P=1.)
base = aPObaseJ(env.T, env.R, env.O, 0.9)
assert base.N == 2
assert base.Q == 1  # 1 observation = 1 state

In [ ]:
# Test Xisa (obs-action policy -> state-action policy)
X = jnp.ones((2, 1, 2)) / 2.0  # uniform obs-action policy (N, Q, M)
Xisa_ = base.Xisa(X)
assert Xisa_.shape == (2, 1, 2)  # (N, Z, M)
assert jnp.allclose(Xisa_.sum(-1), 1.0)

In [ ]:
# Test fast_Bios
bios = base.fast_Bios(X)
assert bios.shape == (2, 1, 1)  # (N, Q, Z)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()